# BagZITboost — Fixed HP + Fixed PP + GroupTargetEncoder

**목적**: HP/전처리 모두 고정하고 GroupTargetEncoder(GTE) 만 추가하여, GTE 의 순수 효과를 측정.

**구성** (둘 다 고정):
- **Model HP**: `bag_zit_pp_hpo` 의 `FIXED_HP` (zit-final-100 best, 180 trial 결과)
- **PP PARAMS**: `bag_zit_hpo` 의 `PARAMS` (origin/main 01_zit_only 동일)
- **τ_π**: off (= 1.0)
- **GTE**: `DEFAULT_GROUP_SPECS` (lot, wafer, wp — 9 encoding) × `alpha=20.0`

**No-cheating 보장**:
- BagZIT CV를 `GroupKFold(n_splits=5)` 로 통일 (`np.unique(ufs_tr)` 위에서 split — GTE 내부 split 과 동일 알고리즘)
- GTE 인코딩은 **각 fold 마다 tr_units 로 재 fit** 하고 tr+vl+val+test 에 lookup → `vl_units` 의 target 은 어떤 GTE 통계에도 사용되지 않음
- val/test 도 fold 별 tr_units fit 으로 인코딩 → 5 fold 평균 (학습/예측 일관)

**격리**: 출력 `4_output/_temp/bag_zit_fixed_ge/`. 기존 모듈/노트북 영향 없음.

## 1. 환경 + import

In [1]:
import os, sys

%run ../../../setup.py

import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from utils.config import (
    PROJECT_ROOT, SEED, TARGET_COL, KEY_COL, DIE_KEY_COL,
    SPLIT_COL, OUTPUT_DIR,
)
from utils.data import load_all, get_feat_cols, split_xs

MODEL_ROOT = os.path.join(PROJECT_ROOT, '3_modeling')
if MODEL_ROOT not in sys.path:
    sys.path.insert(0, MODEL_ROOT)

PP_ROOT = os.path.join(PROJECT_ROOT, '2_preprocessing')
if PP_ROOT not in sys.path:
    sys.path.insert(0, PP_ROOT)

from final.modules import preprocess
from modules.zi_tweedie import ZITboostRegressor
from group_encoder import GroupTargetEncoder, DEFAULT_GROUP_SPECS

import lightgbm as lgb
from sklearn.model_selection import GroupKFold

import logging
logging.getLogger('lightgbm').setLevel(logging.ERROR)

print(f'PROJECT_ROOT = {PROJECT_ROOT}')
print(f'GTE group_specs = {DEFAULT_GROUP_SPECS}')

setup 완료
PROJECT_ROOT = c:\Users\Dell5371\Desktop\기업연계프로젝트
GTE group_specs = [('lot', 'lot_id'), ('wafer', 'wafer_id'), ('wp', ['wafer_id', 'position'])]


## 2. 실험 설정 (HP / PP / GTE 모두 고정)

In [2]:
# ── 실험 식별 ──
EXP_ID = 'bag-zit-fixed-ge-001'
USER   = 'jh'
N_FOLDS = 5
CLIP_Y_EXTREME = True

# ── 출력 경로 ──
OUT_DIR = os.path.join(OUTPUT_DIR, '_temp', 'bag_zit_fixed_ge')
os.makedirs(OUT_DIR, exist_ok=True)

# ── PP PARAMS (bag_zit_hpo 와 동일, origin/main 01_zit_only) ──
PARAMS = {
    'missing_threshold':          0.4,
    'corr_threshold':             0.90,
    'corr_keep_by':               'std',
    'add_indicator':              True,
    'indicator_threshold':        0.05,
    'spatial_max_dist':           5.0,
    'post_impute_corr_threshold': 0.98,
    'post_impute_corr_keep_by':   'std',
}

# ── FIXED_HP (bag_zit_pp_hpo 와 동일, zit-final-100 best) ──
FIXED_HP = dict(
    zeta                  = 1.149489889666854,
    n_em_iters            = 15,
    em_tol                = 1e-7,
    mu_n_estimators       = 164,
    mu_learning_rate      = 0.00738594709027388,
    mu_num_leaves         = 153,
    mu_max_depth          = 5,
    mu_min_child_samples  = 72,
    mu_subsample          = 0.6938934613616774,
    mu_colsample_bytree   = 0.35457351273995524,
    mu_reg_alpha          = 0.0001691645198209928,
    mu_reg_lambda         = 0.07334833575531088,
    pi_n_estimators       = 215,
    pi_learning_rate      = 0.07300634153014683,
    pi_num_leaves         = 69,
    pi_max_depth          = 8,
    pi_min_child_samples  = 44,
    phi_n_estimators      = 67,
    phi_learning_rate     = 0.012225381475545456,
    phi_num_leaves        = 72,
    phi_max_depth         = 5,
    phi_min_child_samples = 29,
    random_state          = SEED,
    n_jobs                = -1,
    verbose               = -1,
    device                = 'cpu',
)

# ── GTE 설정 (기본값) ──
GTE_ALPHA   = 20.0
GROUP_SPECS = DEFAULT_GROUP_SPECS  # [('lot','lot_id'), ('wafer','wafer_id'), ('wp',['wafer_id','position'])]

# ── τ_π off ──
TAU_PI = 1.0

print(f'EXP_ID={EXP_ID} | N_FOLDS={N_FOLDS}')
print(f'OUT_DIR={OUT_DIR}')
print(f'PARAMS keys: {list(PARAMS)}')
print(f'FIXED_HP keys: {len(FIXED_HP)}')
print(f'GTE: alpha={GTE_ALPHA}, group_specs={[n for n,_ in GROUP_SPECS]}')
print(f'TAU_PI={TAU_PI} (off)')

EXP_ID=bag-zit-fixed-ge-001 | N_FOLDS=5
OUT_DIR=c:\Users\Dell5371\Desktop\기업연계프로젝트\4_output\_temp\bag_zit_fixed_ge
PARAMS keys: ['missing_threshold', 'corr_threshold', 'corr_keep_by', 'add_indicator', 'indicator_threshold', 'spatial_max_dist', 'post_impute_corr_threshold', 'post_impute_corr_keep_by']
FIXED_HP keys: 26
GTE: alpha=20.0, group_specs=['lot', 'wafer', 'wp']
TAU_PI=1.0 (off)


## 3. 데이터 로드 + GroupTargetEncoder 메타 컬럼 파싱

`run_wf_xy` 에서 `lot_id`, `wafer_id` 파싱 (GTE 입력 + per-fold lookup 에 필요).

In [3]:
xs, ys = load_all()

# GTE 메타 컬럼 (lot_id, wafer_id) 파싱 — 사본 반환
xs = GroupTargetEncoder.parse_group_columns(xs)
print(f'parse_group_columns 후: lot_id={"lot_id" in xs.columns}, '
      f'wafer_id={"wafer_id" in xs.columns}')

feat_cols = get_feat_cols(xs)   # X0~X1086 만 (lot_id/wafer_id 미포함)
xs_dict = split_xs(xs)

ys_input = {k: v.copy() for k, v in ys.items()}
if CLIP_Y_EXTREME:
    y_raw = ys_input['train'][TARGET_COL]
    second_max = y_raw[y_raw < y_raw.max()].max()
    n_clipped = (y_raw >= 1.0).sum()
    ys_input['train'][TARGET_COL] = y_raw.clip(upper=second_max)
    print(f'[CLIP_Y_EXTREME] 1.0 → {second_max:.6f} clip, {n_clipped}개 샘플')

y_train_unit = ys_input['train'].set_index(KEY_COL)[TARGET_COL]
y_val_unit   = ys_input['validation'].set_index(KEY_COL)[TARGET_COL]
y_test_unit  = ys_input['test'].set_index(KEY_COL)[TARGET_COL]

print(f'\n[데이터 로드 완료]')
print(f'  xs: {xs.shape}, feat_cols(X only): {len(feat_cols)}')
print(f'  unit train={len(y_train_unit):,}, val={len(y_val_unit):,}, test={len(y_test_unit):,}')

[load_xs] all-NaN 행 407개 제거 → 174,573행
[load_xs] 4 position 미만 unit 1개 제거 (split별: {'train': 1}) → die 174,573 → 174,572
[load_ys] train: xs에 없는 unit 60개 제거 → 26,187
[load_ys] validation: xs에 없는 unit 22개 제거 → 8,727
[load_ys] test: xs에 없는 unit 20개 제거 → 8,729
Xs: (174572, 1091)  |  Ys: train=26,187, val=8,727, test=8,729
parse_group_columns 후: lot_id=True, wafer_id=True
[CLIP_Y_EXTREME] 1.0 → 0.097417 clip, 1개 샘플

[데이터 로드 완료]
  xs: (174572, 1093), feat_cols(X only): 1087
  unit train=26,187, val=8,727, test=8,729


## 4. 전처리 (preprocess.run, PARAMS 고정)

`feat_cols` 에 X 컬럼만 포함 → cleaning은 X 만 처리. `lot_id`/`wafer_id` 는 비-feature 메타 컬럼으로 통과 (cleaning이 건드리지 않음).

전처리 후 `xs_train_die`/`val`/`test` 에 `lot_id`/`wafer_id` 가 없으면 reindex 로 재부착.

In [4]:
pp = preprocess.run(xs, ys_input, feat_cols, xs_dict, params=PARAMS)
xs_train_die = pp['xs_train']
xs_val_die   = pp['xs_val']
xs_test_die  = pp['xs_test']
feat_cols_clean = pp['feat_cols']   # cleaning 후 X feature + indicator

# lot_id / wafer_id / position 가 cleaning 후 살아있는지 확인 후 없으면 reindex 로 부착
META_COLS = ['lot_id', 'wafer_id', 'position']
for split_name, df in [('train', xs_train_die), ('val', xs_val_die), ('test', xs_test_die)]:
    for col in META_COLS:
        if col not in df.columns:
            df[col] = xs[col].reindex(df.index).values
            print(f'  [meta reattach] {split_name}/{col} ← reindex from xs')

print(f'\n[전처리 완료]')
print(f'  feat_cols_clean (X+indicator): {len(feat_cols_clean)}')
print(f'  xs_train_die: {xs_train_die.shape}')
print(f'  xs_val_die  : {xs_val_die.shape}')
print(f'  xs_test_die : {xs_test_die.shape}')

# numpy 변환 (X 부분만 — GTE 부분은 fold별로 추가)
X_train_X_only = xs_train_die[feat_cols_clean].values.astype(np.float64)
X_val_X_only   = xs_val_die[feat_cols_clean].values.astype(np.float64)
X_test_X_only  = xs_test_die[feat_cols_clean].values.astype(np.float64)
uid_train_die = xs_train_die[KEY_COL].values
uid_val_die   = xs_val_die[KEY_COL].values
uid_test_die  = xs_test_die[KEY_COL].values
y_train_die_broadcast = pd.Series(uid_train_die).map(y_train_unit).values.astype(np.float64)
assert not pd.isna(y_train_die_broadcast).any(), 'unmapped train die y 존재'

n_train_die, n_val_die, n_test_die = len(X_train_X_only), len(X_val_X_only), len(X_test_X_only)
print(f'  X_train_X_only: {X_train_X_only.shape}, X_val: {X_val_X_only.shape}, X_test: {X_test_X_only.shape}')

[Stage 0] 웨이퍼맵 사전 제외: 1087 → 1033 (54개 제거)
클리닝 파이프라인 시작
원본 feature 수: 1033
[상수/극저분산 제거] threshold=1e-06
  제거: 105개, 잔여: 928개
    컬럼: 1033 → 928 (105개 제거)
    DataFrame: (104748, 988)

[고결측 제거] threshold=40%
  제거: 5개, 잔여: 923개
    컬럼: 928 → 923 (5개 제거)
    DataFrame: (104748, 983)

[중복 컬럼 제거] sample_n=5000
  제거: 27개, 잔여: 896개
    컬럼: 923 → 896 (27개 제거)
    DataFrame: (104748, 956)

[고상관 제거] threshold=0.9, keep_by=std (std)
  제거: 332개, 잔여: 564개
    컬럼: 896 → 564 (332개 제거)
    DataFrame: (104748, 624)

[결측 indicator] 9개 컬럼 추가 (결측률 >= 5%)
[공간 보간 imputation] 총 결측: 343,494
  train-only 모드: train 104,748 / 전체 174,572 행
  1단계 (공간 보간, dist<=5.0): 156,772개 채움 → 잔여: 186,722
  2단계 (lot 평균, train 기준): 105,526개 채움 → 잔여: 81,196
  3단계 (train 전체 평균): 81,196개 채움 → 잔여: 0

  [요약] 343,494 → 공간(156,772) → lot(105,526) → 전체(81,196) → 잔여(0)

[고상관 제거] threshold=0.98, keep_by=std (std)
  제거: 0개, 잔여: 564개
    [고상관 제거 2차 / imputation 후] threshold=0.98
    컬럼: 564 → 564 (0개 제거)
    DataFrame: (104748, 633)

클리닝 완료

## 5. BagZITboostRegressor 정의 (기존 노트북과 동일)

In [5]:
class BagZITboostRegressor(ZITboostRegressor):
    """ZITboost + bag (unit) constraint via B3 allocation."""

    @staticmethod
    def _allocate_b3(unit_y_per_unit, contribution, inverse, n_units):
        contrib_sum_per_unit = np.zeros(n_units)
        np.add.at(contrib_sum_per_unit, inverse, contribution)
        contrib_sum_die = contrib_sum_per_unit[inverse]
        n_die_per_unit = np.bincount(inverse, minlength=n_units).astype(np.float64)
        n_die_die = n_die_per_unit[inverse]
        share = np.where(
            contrib_sum_die > 1e-12,
            contribution / np.maximum(contrib_sum_die, 1e-12),
            1.0 / np.maximum(n_die_die, 1.0),
        )
        return unit_y_per_unit[inverse] * share

    def fit(self, X, y, unit_id):
        X = np.asarray(X, dtype=np.float64)
        y = np.asarray(y, dtype=np.float64).ravel()
        unit_id = np.asarray(unit_id)

        unique_units, first_idx, inverse = np.unique(
            unit_id, return_index=True, return_inverse=True
        )
        n_units = len(unique_units)
        unit_y_per_unit = y[first_idx]

        n_die_per_unit = np.bincount(inverse, minlength=n_units).astype(np.float64)
        y_die_alloc = unit_y_per_unit[inverse] / np.maximum(n_die_per_unit[inverse], 1.0)

        pi_arr, mu_arr, phi_arr = self._initialize(X, y_die_alloc)
        self._phi_current = phi_arr

        self.em_history_ = []
        prev_rmse = np.inf
        em_iter = 0

        for em_iter in range(self.n_em_iters):
            if em_iter > 0:
                contribution = np.clip((1 - pi_arr) * mu_arr, 0, None)
                y_die_alloc = self._allocate_b3(
                    unit_y_per_unit, contribution, inverse, n_units
                )

            posterior = self._e_step(y_die_alloc, pi_arr, mu_arr, phi_arr)

            self._phi_current = phi_arr
            lgb_pi, lgb_mu, lgb_phi, pi_arr, mu_arr, phi_arr = \
                self._m_step(X, y_die_alloc, posterior)

            pred_die = np.clip((1 - pi_arr) * mu_arr, 0, None)
            pred_unit = np.zeros(n_units)
            np.add.at(pred_unit, inverse, pred_die)
            rmse_unit = float(np.sqrt(np.mean((unit_y_per_unit - pred_unit) ** 2)))

            self.em_history_.append({
                'iter':      em_iter + 1,
                'unit_rmse': rmse_unit,
                'pi_mean':   float(pi_arr.mean()),
                'mu_mean':   float(mu_arr.mean()),
            })

            rmse_delta = prev_rmse - rmse_unit
            if em_iter >= 2 and abs(rmse_delta) < self.em_tol:
                break
            prev_rmse = rmse_unit

        self.n_em_iters_actual_ = em_iter + 1
        self.lgb_pi_ = lgb_pi
        self.lgb_mu_ = lgb_mu
        self.lgb_phi_ = lgb_phi
        self.fitted_ = True
        return self


print('BagZITboostRegressor 정의 완료')

BagZITboostRegressor 정의 완료


## 6. GroupKFold split + GTE per-fold lookup helpers

**Split 정렬**: `np.unique(ufs_tr)` 위에서 `GroupKFold` — `GroupTargetEncoder` 내부 split 과 동일 알고리즘 (deterministic, shuffle 없음).

**Per-fold GTE**: 각 fold 의 `tr_units` 만으로 group statistics fit → tr+vl+val+test 에 lookup 적용. `vl_units` 타깃은 어떤 통계에도 들어가지 않음 (no cheating).

In [6]:
# ── fold split (GroupKFold, deterministic — GTE 내부 split 과 동일 알고리즘) ──
ufs_full_tr = xs[xs[SPLIT_COL] == 'train'][KEY_COL].values
unit_ids_train_unique = np.unique(ufs_full_tr)   # sorted, GTE 내부와 일치
gkf_global = GroupKFold(n_splits=N_FOLDS)
FOLDS = list(gkf_global.split(
    np.zeros(len(unit_ids_train_unique)),
    groups=unit_ids_train_unique,
))
print(f'fold split: {N_FOLDS} folds, {len(FOLDS)} 개')
print(f'  unit_ids_train_unique: {len(unit_ids_train_unique):,} (sorted)')

# ── per-fold GTE helpers ──
GE_COL_NAMES = []
for name, _ in GROUP_SPECS:
    GE_COL_NAMES.extend([f'{name}_te', f'{name}_zero_rate', f'{name}_pos_mean'])
print(f'\nGE_COL_NAMES ({len(GE_COL_NAMES)}): {GE_COL_NAMES}')

# 사전 pids 계산 (split 별, group spec 별) — fold loop 마다 재계산하지 않음
pids_per_spec = {}
for name, col_or_cols in GROUP_SPECS:
    pids_per_spec[name] = {
        'train': GroupTargetEncoder._pids_for_spec(xs_train_die, col_or_cols),
        'val':   GroupTargetEncoder._pids_for_spec(xs_val_die,   col_or_cols),
        'test':  GroupTargetEncoder._pids_for_spec(xs_test_die,  col_or_cols),
    }
print('pids_per_spec 사전계산 완료')


def _fold_lookup_for_spec(pids_train_full, y_die_full, tr_die_mask, alpha):
    """tr_die_mask 만으로 fit → (te_dict, zero_dict, pos_dict, gm_mean, gm_zero, gm_pos)."""
    train_pids = pids_train_full[tr_die_mask]
    train_y    = y_die_full[tr_die_mask]
    is_zero = (train_y == 0).astype(np.float64)
    gm_mean = float(train_y.mean())
    gm_zero = float(is_zero.mean())
    pos_mask = train_y > 0
    gm_pos = float(train_y[pos_mask].mean()) if pos_mask.sum() > 0 else 0.0

    te_d   = GroupTargetEncoder._smoothed_mean(train_pids, train_y, gm_mean, alpha)
    zero_d = GroupTargetEncoder._smoothed_mean(train_pids, is_zero, gm_zero, alpha)
    if pos_mask.sum() > 0:
        pos_d = GroupTargetEncoder._smoothed_mean(
            train_pids[pos_mask], train_y[pos_mask], gm_pos, alpha
        )
    else:
        pos_d = {}
    return te_d, zero_d, pos_d, gm_mean, gm_zero, gm_pos


def _apply_lookup(pids, te_d, zero_d, pos_d, gm_mean, gm_zero, gm_pos):
    s = pd.Series(pids)
    out_te   = s.map(te_d).fillna(gm_mean).values
    out_zero = s.map(zero_d).fillna(gm_zero).values
    out_pos  = s.map(pos_d).fillna(gm_pos).values
    return out_te, out_zero, out_pos


def build_fold_ge_matrices(tr_die_mask, alpha):
    """한 fold 의 tr_die_mask 로 fit 후, train/val/test 모든 행에 lookup.

    Returns (train_ge, val_ge, test_ge) — 각 (n_die, len(GE_COL_NAMES)) np.ndarray.
    """
    train_arrs, val_arrs, test_arrs = [], [], []
    for name, _ in GROUP_SPECS:
        te_d, zero_d, pos_d, gm_m, gm_z, gm_p = _fold_lookup_for_spec(
            pids_per_spec[name]['train'], y_train_die_broadcast, tr_die_mask, alpha
        )
        for split_key, sink in [('train', train_arrs), ('val', val_arrs), ('test', test_arrs)]:
            te, ze, po = _apply_lookup(
                pids_per_spec[name][split_key], te_d, zero_d, pos_d, gm_m, gm_z, gm_p
            )
            sink.extend([te, ze, po])
    return (
        np.column_stack(train_arrs),
        np.column_stack(val_arrs),
        np.column_stack(test_arrs),
    )


# 검증 — fold 0 에 대해 ge matrix 한 번 만들어 NaN/모양 확인
_tr_units = unit_ids_train_unique[FOLDS[0][0]]
_tr_mask = np.isin(uid_train_die, _tr_units)
_g_tr, _g_va, _g_te = build_fold_ge_matrices(_tr_mask, GTE_ALPHA)
print(f'\n[fold 0 GTE matrix 검증]')
print(f'  train_ge: {_g_tr.shape}, NaN={np.isnan(_g_tr).any()}')
print(f'  val_ge  : {_g_va.shape}, NaN={np.isnan(_g_va).any()}')
print(f'  test_ge : {_g_te.shape}, NaN={np.isnan(_g_te).any()}')
del _g_tr, _g_va, _g_te, _tr_mask, _tr_units

fold split: 5 folds, 5 개
  unit_ids_train_unique: 26,187 (sorted)

GE_COL_NAMES (9): ['lot_te', 'lot_zero_rate', 'lot_pos_mean', 'wafer_te', 'wafer_zero_rate', 'wafer_pos_mean', 'wp_te', 'wp_zero_rate', 'wp_pos_mean']
pids_per_spec 사전계산 완료

[fold 0 GTE matrix 검증]
  train_ge: (104748, 9), NaN=False
  val_ge  : (34908, 9), NaN=False
  test_ge : (34916, 9), NaN=False


## 7. 5-fold refit + die-level π/μ/pred 캡처

각 fold 마다:
1. `tr_units` 로 GTE fit → train/val/test 전체에 lookup
2. X (cleaned `feat_cols_clean`) + GE (9 cols) 로 학습 데이터 구성
3. `BagZIT(FIXED_HP).fit(X_tr, y_tr, unit_id=uid_tr)`
4. OOF (`vl_units`), val, test 모두 die-level π/μ/pred 캡처
5. val/test 는 5 fold 평균

In [7]:
import time

# die-level 캡처 버퍼
oof_die_pi   = np.full(n_train_die, np.nan)
oof_die_mu   = np.full(n_train_die, np.nan)
oof_die_pred = np.full(n_train_die, np.nan)

val_die_pi   = np.zeros(n_val_die)
val_die_mu   = np.zeros(n_val_die)
val_die_pred = np.zeros(n_val_die)

test_die_pi   = np.zeros(n_test_die)
test_die_mu   = np.zeros(n_test_die)
test_die_pred = np.zeros(n_test_die)

print(f'=== 5-fold refit (FIXED_HP + per-fold GTE lookup) ===')
t0 = time.time()
for fold_idx, (tr_uidx, vl_uidx) in enumerate(FOLDS):
    tr_units = unit_ids_train_unique[tr_uidx]
    vl_units = unit_ids_train_unique[vl_uidx]
    tr_die_mask = np.isin(uid_train_die, tr_units)
    vl_die_mask = np.isin(uid_train_die, vl_units)

    # fold-local GTE matrix (train/val/test) — tr_units 만으로 fit, lookup 적용
    train_ge, val_ge, test_ge = build_fold_ge_matrices(tr_die_mask, GTE_ALPHA)

    # X + GE concat
    X_train_full = np.concatenate([X_train_X_only, train_ge], axis=1)
    X_val_full   = np.concatenate([X_val_X_only,   val_ge],   axis=1)
    X_test_full  = np.concatenate([X_test_X_only,  test_ge],  axis=1)

    X_tr   = X_train_full[tr_die_mask]
    X_vl   = X_train_full[vl_die_mask]
    y_tr   = y_train_die_broadcast[tr_die_mask]
    uid_tr = uid_train_die[tr_die_mask]

    model = BagZITboostRegressor(**FIXED_HP)
    model.fit(X_tr, y_tr, unit_id=uid_tr)

    # OOF (vl_units)
    pi_vl, mu_vl, _ = model.predict_components(X_vl)
    pred_vl = np.clip((1 - pi_vl) * mu_vl, 0, None)
    oof_die_pi[vl_die_mask]   = pi_vl
    oof_die_mu[vl_die_mask]   = mu_vl
    oof_die_pred[vl_die_mask] = pred_vl

    # val/test (5-fold avg)
    pi_v, mu_v, _ = model.predict_components(X_val_full)
    pi_t, mu_t, _ = model.predict_components(X_test_full)
    val_die_pi    += pi_v / N_FOLDS
    val_die_mu    += mu_v / N_FOLDS
    val_die_pred  += np.clip((1 - pi_v) * mu_v, 0, None) / N_FOLDS
    test_die_pi   += pi_t / N_FOLDS
    test_die_mu   += mu_t / N_FOLDS
    test_die_pred += np.clip((1 - pi_t) * mu_t, 0, None) / N_FOLDS

    print(f'  fold {fold_idx+1}/{N_FOLDS} done ({time.time()-t0:.0f}s)')

assert not np.isnan(oof_die_pi).any(),   'OOF die π 미커버'
assert not np.isnan(oof_die_mu).any(),   'OOF die μ 미커버'
assert not np.isnan(oof_die_pred).any(), 'OOF die pred 미커버'

print(f'\n[refit 완료] die-level π/μ/pred 캡처 OK ({time.time()-t0:.0f}s)')
print(f'  oof_die: {n_train_die}, val_die: {n_val_die}, test_die: {n_test_die}')

=== 5-fold refit (FIXED_HP + per-fold GTE lookup) ===
  fold 1/5 done (230s)
  fold 2/5 done (454s)
  fold 3/5 done (652s)
  fold 4/5 done (848s)
  fold 5/5 done (1035s)

[refit 완료] die-level π/μ/pred 캡처 OK (1035s)
  oof_die: 104748, val_die: 34908, test_die: 34916


## 8. unit aggregate + RMSE (τ_π = 1.0 → raw 만)

In [8]:
def _sum_die_to_unit(pred_die, uid_die):
    unit_id = np.asarray(uid_die)
    unique_units, inverse = np.unique(unit_id, return_inverse=True)
    n_units = len(unique_units)
    pred_unit = np.zeros(n_units)
    np.add.at(pred_unit, inverse, pred_die)
    return pred_unit, unique_units


# unit aggregate
oof_unit_arr,  oof_unit_ids  = _sum_die_to_unit(oof_die_pred,  uid_train_die)
val_unit_arr,  val_unit_ids  = _sum_die_to_unit(val_die_pred,  uid_val_die)
test_unit_arr, test_unit_ids = _sum_die_to_unit(test_die_pred, uid_test_die)

oof_unit  = pd.Series(oof_unit_arr,  index=oof_unit_ids).reindex(y_train_unit.index)
val_unit  = pd.Series(val_unit_arr,  index=val_unit_ids).reindex(y_val_unit.index)
test_unit = pd.Series(test_unit_arr, index=test_unit_ids).reindex(y_test_unit.index)


def _rmse(pred, true):
    return float(np.sqrt(np.mean((pred.values - true.values) ** 2)))


oof_rmse  = _rmse(oof_unit,  y_train_unit)
val_rmse  = _rmse(val_unit,  y_val_unit)
test_rmse = _rmse(test_unit, y_test_unit)

print('=' * 75)
print(f'  Fixed HP + Fixed PP + GTE (alpha={GTE_ALPHA}, {len(GE_COL_NAMES)} cols)')
print('=' * 75)
print(f'  {"":12s}  {"OOF":>11s}  {"val":>11s}  {"test":>11s}')
print(f'  {"RMSE":12s}  {oof_rmse:11.6f}  {val_rmse:11.6f}  {test_rmse:11.6f}')
print('-' * 75)
print(f'  비교 기준선 (참조):')
print(f'    BagZIT baseline (HP+PP fixed, GTE 없음): OOF=0.005524, val=0.005729, test=0.008428')
print(f'    ZIT 단독 zit-final-100:                 OOF=0.005501')
print(f'    bag_zit_pp_hpo trial 0 (no GTE):        OOF=0.005501, val=0.005709, test=0.008412')
print('=' * 75)

  Fixed HP + Fixed PP + GTE (alpha=20.0, 9 cols)
                        OOF          val         test
  RMSE             0.005514     0.005717     0.008412
---------------------------------------------------------------------------
  비교 기준선 (참조):
    BagZIT baseline (HP+PP fixed, GTE 없음): OOF=0.005524, val=0.005729, test=0.008428
    ZIT 단독 zit-final-100:                 OOF=0.005501
    bag_zit_pp_hpo trial 0 (no GTE):        OOF=0.005501, val=0.005709, test=0.008412


## 9. 아티팩트 저장 (die + unit, meta)

In [9]:
import json

def _build_die_df(uid_arr, die_id_arr, position_arr, pi, mu, pred, y_unit):
    df = pd.DataFrame({
        KEY_COL:        uid_arr,
        DIE_KEY_COL:    die_id_arr,
        'position':     position_arr,
        'pi':           pi,
        'one_minus_pi': 1.0 - pi,
        'mu':           mu,
        'pred':         pred,
    })
    if y_unit is not None:
        df[TARGET_COL] = df[KEY_COL].map(y_unit)
    return df


oof_die_df = _build_die_df(
    uid_train_die, xs_train_die[DIE_KEY_COL].values, xs_train_die['position'].values,
    oof_die_pi, oof_die_mu, oof_die_pred, y_train_unit,
)
val_die_df = _build_die_df(
    uid_val_die, xs_val_die[DIE_KEY_COL].values, xs_val_die['position'].values,
    val_die_pi, val_die_mu, val_die_pred, y_val_unit,
)
test_die_df = _build_die_df(
    uid_test_die, xs_test_die[DIE_KEY_COL].values, xs_test_die['position'].values,
    test_die_pi, test_die_mu, test_die_pred, y_test_unit,
)
oof_die_df.to_csv(os.path.join(OUT_DIR,  'oof_die.csv'),  index=False)
val_die_df.to_csv(os.path.join(OUT_DIR,  'val_die.csv'),  index=False)
test_die_df.to_csv(os.path.join(OUT_DIR, 'test_die.csv'), index=False)


def _build_unit_df(unit_pred, y_unit):
    return pd.DataFrame({
        KEY_COL:  unit_pred.index.values,
        'pred':   unit_pred.values,
        'health': y_unit.reindex(unit_pred.index).values,
    })


_build_unit_df(oof_unit,  y_train_unit).to_csv(os.path.join(OUT_DIR, 'oof_unit.csv'),  index=False)
_build_unit_df(val_unit,  y_val_unit ).to_csv(os.path.join(OUT_DIR, 'val_unit.csv'),  index=False)
_build_unit_df(test_unit, y_test_unit).to_csv(os.path.join(OUT_DIR, 'test_unit.csv'), index=False)

# meta — Optuna 없이 고정 1회 학습이므로 best_params 대신 meta 한 파일에 정리
meta = {
    'exp_id':              EXP_ID,
    'model':               'BagZITboost (FIXED_HP, FIXED_PP) + GroupTargetEncoder',
    'fold_split':          'GroupKFold(n_splits=5) over np.unique(ufs_tr)',
    'gte_strategy':        'per-fold lookup (fit on tr_units only, apply to tr+vl+val+test)',
    'tau_pi':              TAU_PI,
    'gte_alpha':           GTE_ALPHA,
    'gte_group_specs':     [n for n, _ in GROUP_SPECS],
    'gte_n_cols':          len(GE_COL_NAMES),
    'gte_col_names':       GE_COL_NAMES,
    'n_folds':             N_FOLDS,
    'oof_rmse':            oof_rmse,
    'val_rmse':            val_rmse,
    'test_rmse':           test_rmse,
    'preprocess_PARAMS':   PARAMS,
    'effective_pp_params': pp['effective_params'],
    'fixed_hp':            {k: v for k, v in FIXED_HP.items()
                            if k not in ['random_state', 'n_jobs', 'verbose', 'device']},
    'fixed_hp_source':     '4_output/final/zit_only/best_params.json (zit-final-100 best)',
    'pp_params_source':    'origin/main 01_zit_only / bag_zit_hpo PARAMS',
    'CLIP_Y_EXTREME':      CLIP_Y_EXTREME,
    'feat_cols_clean_n':   len(feat_cols_clean),
    'X_plus_ge_n':         len(feat_cols_clean) + len(GE_COL_NAMES),
    'SEED':                int(SEED),
}
with open(os.path.join(OUT_DIR, 'meta.json'), 'w', encoding='utf-8') as f:
    json.dump(meta, f, indent=2, ensure_ascii=False, default=str)

print(f'저장 완료: {OUT_DIR}')
for f_ in sorted(os.listdir(OUT_DIR)):
    sz = os.path.getsize(os.path.join(OUT_DIR, f_)) / 1024
    print(f'  {f_:35s}  {sz:10,.1f} KB')

저장 완료: c:\Users\Dell5371\Desktop\기업연계프로젝트\4_output\_temp\bag_zit_fixed_ge
  meta.json                                   2.5 KB
  oof_die.csv                            11,999.2 KB
  oof_unit.csv                              971.8 KB
  test_die.csv                            3,998.7 KB
  test_unit.csv                             323.9 KB
  val_die.csv                             3,997.8 KB
  val_unit.csv                              323.9 KB


## 10. 요약

In [10]:
print('=' * 75)
print(' BagZITboost — Fixed HP + Fixed PP + GroupTargetEncoder — 결과 요약')
print('=' * 75)
print(f'  EXP_ID            : {EXP_ID}')
print(f'  Fold split        : GroupKFold (deterministic, GTE 와 동일 알고리즘)')
print(f'  GTE strategy      : per-fold lookup (no cheating)')
print(f'  GTE group_specs   : {[n for n, _ in GROUP_SPECS]}')
print(f'  GTE alpha         : {GTE_ALPHA}')
print(f'  GTE cols          : {len(GE_COL_NAMES)} ({GE_COL_NAMES})')
print(f'  τ_π               : {TAU_PI} (off)')
print(f'  feat (X clean)    : {len(feat_cols_clean)}')
print(f'  feat total (X+GE) : {len(feat_cols_clean) + len(GE_COL_NAMES)}')
print('-' * 75)
print(f'  {"":10s}  {"OOF":>11s}  {"val":>11s}  {"test":>11s}')
print(f'  {"RMSE":10s}  {oof_rmse:11.6f}  {val_rmse:11.6f}  {test_rmse:11.6f}')
print('-' * 75)
print(f'  비교 기준선:')
print(f'    BagZIT baseline (HP+PP fixed, no GTE): OOF=0.005524, val=0.005729, test=0.008428')
print(f'    bag_zit_pp_hpo trial 0 (no GTE):       OOF=0.005501, val=0.005709, test=0.008412')
print(f'    ZIT 단독 zit-final-100:                OOF=0.005501')
print('=' * 75)
print(f'  → GTE 추가 후 RMSE 가 baseline 대비 감소하면 group encoding 효과 입증.')

 BagZITboost — Fixed HP + Fixed PP + GroupTargetEncoder — 결과 요약
  EXP_ID            : bag-zit-fixed-ge-001
  Fold split        : GroupKFold (deterministic, GTE 와 동일 알고리즘)
  GTE strategy      : per-fold lookup (no cheating)
  GTE group_specs   : ['lot', 'wafer', 'wp']
  GTE alpha         : 20.0
  GTE cols          : 9 (['lot_te', 'lot_zero_rate', 'lot_pos_mean', 'wafer_te', 'wafer_zero_rate', 'wafer_pos_mean', 'wp_te', 'wp_zero_rate', 'wp_pos_mean'])
  τ_π               : 1.0 (off)
  feat (X clean)    : 573
  feat total (X+GE) : 582
---------------------------------------------------------------------------
                      OOF          val         test
  RMSE           0.005514     0.005717     0.008412
---------------------------------------------------------------------------
  비교 기준선:
    BagZIT baseline (HP+PP fixed, no GTE): OOF=0.005524, val=0.005729, test=0.008428
    bag_zit_pp_hpo trial 0 (no GTE):       OOF=0.005501, val=0.005709, test=0.008412
    ZIT 단독 zit-final-100: 